# Regex Capture Groups and Extraction

## Overview
Capture groups `(...)` allow extracting specific parts of a matching string. In PostgreSQL, the primary functions are:

| Function | Returns | Use case |
|---|---|---|
| `regexp_match(col, pattern)` | `TEXT[]` of first match's groups (or NULL) | Extract one match |
| `regexp_matches(col, pattern, flags)` | `SETOF TEXT[]` (one row per match) | Extract all matches |
| `regexp_replace(col, pattern, replacement)` | `TEXT` | Replace matched text |

**Capture group syntax:**
```sql
-- Pattern: (\d+)/(\d+)\s*mmHg
-- Applied to: 'BP 142/88 mmHg'
-- Result: ARRAY['142', '88']
-- Access: (regexp_match(col, '(\d+)/(\d+)'))[1]  → '142'
--         (regexp_match(col, '(\d+)/(\d+)'))[2]  → '88'
```

**PostgreSQL backreferences in regexp_replace:**
```sql
regexp_replace('10 mg', '(\d+)\s+(mg)', '')  → '10mg'  (groups  and  concatenated)
```

**SQLite note:** These notebooks use Python's `re` module in a loop to demonstrate extraction. The PostgreSQL function calls are shown in every section.

---

In [ ]:
import sqlite3, re

def make_conn():
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    conn.create_function("regexp", 2,
        lambda p, s: bool(re.search(p, s or "", re.IGNORECASE)))
    conn.create_function("regexp_replace", 3,
        lambda p, s, r: re.sub(p, r, s or "", flags=re.IGNORECASE))
    conn.create_function("regexp_extract", 2,
        lambda p, s: (re.search(p, s or "", re.IGNORECASE) or type("", (), {"group": lambda self, n=0: None})()).group(0))
    conn.create_function("regexp_extract_g", 3,
        lambda p, s, g: (re.search(p, s or "", re.IGNORECASE) or type("", (), {"group": lambda self, n=0: None})()).group(g))
    return conn

conn = make_conn()
conn.executescript("""
CREATE TABLE clinical_notes (
    note_id INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id TEXT NOT NULL, note_date TEXT NOT NULL,
    note_text TEXT NOT NULL
);
CREATE TABLE medications (
    med_id INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id TEXT NOT NULL, drug_name TEXT NOT NULL,
    dosage TEXT, frequency TEXT
);
CREATE TABLE lab_results (
    lab_id INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id TEXT NOT NULL, test_name TEXT NOT NULL,
    raw_value TEXT NOT NULL, result_date TEXT NOT NULL
);
INSERT INTO clinical_notes VALUES
    (NULL,'P001','2023-04-10','BP 142/88 mmHg. Prescribed Lisinopril 10mg daily.'),
    (NULL,'P001','2023-10-01','BP improved to 128/82 mmHg. HbA1c 7.2%. Added Metformin 500mg BID.'),
    (NULL,'P002','2023-05-15','SpO2 94%. Salbutamol 2.5mg nebulised. Fluticasone 250mcg/dose.'),
    (NULL,'P003','2023-08-20','BMI 34.2. BP 148/92 mmHg. Consider ACE inhibitor.'),
    (NULL,'P005','2023-11-10','HbA1c 8.9%. Insulin glargine 20 units nocte. Metformin 1g BD.');
INSERT INTO medications VALUES
    (NULL,'P001','Lisinopril','10mg','once daily'),
    (NULL,'P001','Metformin','500mg','BID'),
    (NULL,'P002','Fluticasone','250mcg','BD'),
    (NULL,'P003','Amlodipine','5 mg','once daily'),
    (NULL,'P005','Metformin','1g','BD'),
    (NULL,'P005','Insulin glargine','20 units','nocte'),
    (NULL,'P001','Atorvastatin','40MG','once daily'),
    (NULL,'P003','Ramipril','2.5 mg/day','once daily');
INSERT INTO lab_results VALUES
    (NULL,'P001','HbA1c','7.2%','2023-10-01'),
    (NULL,'P001','eGFR','>60 mL/min','2023-10-01'),
    (NULL,'P002','SpO2','94%','2023-05-15'),
    (NULL,'P005','HbA1c','8.9 %','2023-11-10'),
    (NULL,'P005','HbA1c','<5.7%','2022-01-01'),
    (NULL,'P001','BP','142/88 mmHg','2023-04-10'),
    (NULL,'P001','BP','128/82 mmHg','2023-10-01');
""")
conn.commit()
print("Dataset ready:", 
      conn.execute("SELECT COUNT(*) FROM clinical_notes").fetchone()[0], "notes,",
      conn.execute("SELECT COUNT(*) FROM medications").fetchone()[0], "medications,",
      conn.execute("SELECT COUNT(*) FROM lab_results").fetchone()[0], "lab results")


---
## Extracting BP readings from clinical notes

In [ ]:
import re

# ── Python re for extraction (simulating PostgreSQL regexp_match) ─
print("=== Extracting BP readings from clinical notes ===")
print("PostgreSQL: regexp_match(note_text, '([0-9]+)/([0-9]+)\\s*mmHg')")
print("SQLite: Python re.search in a loop (shown below)")
print()

notes = conn.execute("SELECT note_id, patient_id, note_text FROM clinical_notes").fetchall()
print(f"  {'note_id':<8s}  {'patient_id':<12s}  {'systolic':<10s}  {'diastolic':<10s}  Extracted from")
print("  " + "-"*70)
for note in notes:
    m = re.search(r'(\d+)/(\d+)\s*mmHg', note["note_text"], re.IGNORECASE)
    if m:
        print(f"  {note['note_id']:<8d}  {note['patient_id']:<12s}  "
              f"{m.group(1):<10s}  {m.group(2):<10s}  ...{note['note_text'][max(0,m.start()-10):m.end()+10]}...")

print()
pg_equiv = """
-- PostgreSQL regexp_match: returns an array of capture groups
SELECT note_id, patient_id,
       (regexp_match(note_text, '(\d+)/(\d+)\s*mmHg'))[1]::int AS systolic,
       (regexp_match(note_text, '(\d+)/(\d+)\s*mmHg'))[2]::int AS diastolic
FROM   clinical_notes
WHERE  note_text ~ '\d+/\d+\s*mmHg';

-- regexp_matches: returns SET OF arrays (one row per match if using 'g' flag)
SELECT note_id,
       m[1] AS drug_name
FROM   clinical_notes,
       regexp_matches(note_text, '([A-Z][a-z]+(?:in|ol|ide|ate|ine|pril))', 'g') AS m;
"""
print("PostgreSQL equivalents:")
print(pg_equiv)


---
## regexp_replace: normalising dosage strings

In [ ]:
print("=== regexp_replace: normalising dosage strings ===")

# Show raw dosage values
print("Raw dosage values:")
rows = conn.execute("SELECT drug_name, dosage FROM medications ORDER BY drug_name").fetchall()
for r in rows:
    print(f"  {r['drug_name']:<20s}  {r['dosage']!r}")

print()
# Normalise: remove spaces between number and unit, lowercase unit
print("Normalised dosage (remove space between number and unit, lowercase unit):")
rows = conn.execute(r"""
    SELECT drug_name, dosage,
           regexp_replace(
               regexp_replace(dosage, ' (mg|mcg|g|units)', '\1'),
               '(mg|mcg|g|units)', LOWER('mg')
           ) AS normalised
    FROM medications
    ORDER BY drug_name
""").fetchall()

# Use Python re directly for accurate demo
import re
for r in rows:
    raw = r["dosage"] or ""
    # Step 1: remove space between number and unit
    step1 = re.sub(r'(\d)\s+(mg|mcg|g|units|MG)', r'', raw, flags=re.IGNORECASE)
    # Step 2: lowercase unit
    step2 = re.sub(r'(mg|mcg|g|units|MG)', lambda m: m.group(0).lower(), step1, flags=re.IGNORECASE)
    print(f"  {r['drug_name']:<20s}  {raw!r:<12s}  →  {step2!r}")

print()
pg_replace = """
-- PostgreSQL: normalise space between number and unit, then lowercase
SELECT drug_name, dosage,
       LOWER(
           regexp_replace(dosage, '(\d)\s+(mg|mcg|g|units)', '\1\2', 'i')
       ) AS normalised_dosage
FROM medications;

-- Multiple replacements (chain regexp_replace calls):
SELECT drug_name,
       regexp_replace(
           regexp_replace(dosage, '\s+', '', 'g'),  -- remove all spaces
           '(MG|MCG|G)',                              -- uppercase unit
           lower('\1'),                              -- → lowercase
           'ig'                                       -- case-insensitive + global
       ) AS clean_dosage
FROM medications;
"""
print("PostgreSQL regexp_replace:")
print(pg_replace)


---
## Flags and extracting all matches

In [ ]:
print("=== regexp_replace flags and all-matches extraction ===")

flags_table = [
    ("g", "Global: replace ALL occurrences (not just first)",     "regexp_replace(col, pat, repl, 'g')"),
    ("i", "Case-insensitive",                                     "regexp_replace(col, pat, repl, 'i')"),
    ("gi","Global + case-insensitive",                            "regexp_replace(col, pat, repl, 'gi')"),
    ("n", "Newline-sensitive: ^ and $ match line boundaries",     "regexp_replace(col, pat, repl, 'n')"),
    ("x", "Extended: allow whitespace/comments in pattern",       "PostgreSQL 15+ only"),
]
print("PostgreSQL regexp_replace flag parameter:")
print(f"  {'Flag':<4s}  {'Meaning':<54s}  Syntax")
print("  " + "-"*75)
for flag, meaning, syntax in flags_table:
    print(f"  {flag:<4s}  {meaning:<54s}  {syntax}")

print()
print("=== Extracting ALL HbA1c values from notes (multiple matches per note) ===")
import re
notes = conn.execute("SELECT note_id, patient_id, note_text FROM clinical_notes").fetchall()
for note in notes:
    matches = re.findall(r'HbA1c\s+([0-9.]+)\s*%', note["note_text"], re.IGNORECASE)
    if matches:
        print(f"  note {note['note_id']} ({note['patient_id']}): HbA1c values = {matches}")

pg_all_matches = """
-- PostgreSQL: regexp_matches with 'g' flag returns one row per match
SELECT note_id, patient_id,
       m[1]::numeric AS hba1c_value
FROM   clinical_notes,
       regexp_matches(note_text, 'HbA1c\s+([0-9.]+)\s*%', 'gi') AS m
ORDER BY note_id;
"""
print()
print("PostgreSQL regexp_matches (all occurrences):")
print(pg_all_matches)


---
## Common Pitfalls

**1. regexp_match returns NULL instead of raising an error on no match**
In PostgreSQL, `regexp_match(col, 'pattern')` returns NULL when there is no match (not an empty array or an error). `(regexp_match(col, pattern))[1]` on a non-matching row returns NULL. Always filter with `WHERE col ~ 'pattern'` before calling `regexp_match()` if you expect all rows to match.

**2. Forgetting capture groups in regexp_match**
`regexp_match(col, '\d+/\d+ mmHg')` with no capture groups returns the entire match as `ARRAY['{match}']`. To extract specific parts (systolic, diastolic), add capture groups: `regexp_match(col, '(\d+)/(\d+) mmHg')` → `ARRAY['{systolic}', '{diastolic}']`.

**3. Using the wrong index for regexp_match results**
PostgreSQL arrays are 1-indexed. `(regexp_match(col, '(\d+)/(\d+)'))[1]` is the first capture group. `[0]` returns NULL (there is no index 0 in a PostgreSQL array). Python's `re.search().group(1)` is also 1-indexed (group(0) is the full match).

**4. regexp_replace without 'g' flag only replacing first occurrence**
`regexp_replace(col, ' ', '-')` replaces only the FIRST space. `regexp_replace(col, ' ', '-', 'g')` replaces ALL spaces. This is a common source of partially-cleaned data that looks correct for simple test cases.

**5. Backreference syntax differences between SQLite/Python and PostgreSQL**
In Python `re.sub()` and PostgreSQL `regexp_replace`, backreferences in the replacement string use `\1` (PostgreSQL) or `\g<1>` (Python). In PostgreSQL, the replacement string uses `\1` through `\9`. In Python, use `r'\1'` (raw string) to avoid double-escaping.

**6. Catastrophic backtracking on unanchored patterns with repetition**
`regexp_match(long_text, '(a+)+b')` can take exponential time on strings with many `a` characters followed by no `b`. Unanchored patterns with nested quantifiers are the main cause of regex denial-of-service. Always anchor patterns where possible and test performance on long strings before deploying.


---
*sql_methods_library - Samantha McGarrigle*